# Baseline Tutorial: Average Model Ranking

This notebook builds a simple baseline.

## What this baseline does
- Learns a global ranking of models from training tasks using mean `model_rank`.
- Reuses that same ranking for every test task.
- Evaluates ranking quality with nDCG@1, nDCG@3, and nDCG@5.


In [1]:
import pandas as pd

## 1) Load Dataset Tables

We load task metadata, model metadata, and task-model relevance labels (ground truth).

In [2]:
df_tasks = pd.read_csv("../data/tasks.csv")
df_tasks.head()

,task_id,task_name,task_source,task_class_count,task_type,text_col,label_col,task_hf_dataset_name,task_hf_subset_name,task_hf_url,task_instruction,task_criticalness
0,1,cardiffnlp/tweet_eval@emoji,hf,20,test,text,label,cardiffnlp/tweet_eval,emoji,https://huggingface.co/datasets/cardiffnlp/twe...,Classify the given social media post into one ...,High Criticalness
1,2,cardiffnlp/tweet_eval@emotion,hf,4,test,text,label,cardiffnlp/tweet_eval,emotion,https://huggingface.co/datasets/cardiffnlp/twe...,Classify the given text samples into one of th...,High Criticalness
2,3,cardiffnlp/tweet_eval@hate,hf,2,test,text,label,cardiffnlp/tweet_eval,hate,https://huggingface.co/datasets/cardiffnlp/twe...,Classify the given text as either 'hate' if it...,High Criticalness
3,4,cardiffnlp/tweet_eval@irony,hf,2,test,text,label,cardiffnlp/tweet_eval,irony,https://huggingface.co/datasets/cardiffnlp/twe...,Classify the given social media text as either...,High Criticalness
4,5,cardiffnlp/tweet_eval@offensive,hf,2,test,text,label,cardiffnlp/tweet_eval,offensive,https://huggingface.co/datasets/cardiffnlp/twe...,Classify each social media comment as either '...,Medium Criticalness


In [3]:
df_models = pd.read_csv("../data/models.csv")
df_models.head()

,model_id,model_name,model_hf_url
0,1,GroNLP/hateBERT,https://huggingface.co/GroNLP/hateBERT
1,2,ProsusAI/finbert,https://huggingface.co/ProsusAI/finbert
2,3,albert-base-v2,https://huggingface.co/albert-base-v2
3,4,bert-base-multilingual-uncased,https://huggingface.co/bert-base-multilingual-...
4,5,bert-base-uncased,https://huggingface.co/bert-base-uncased


In [4]:
df_ground_truth = pd.read_csv("../data/ground-truth.csv")
df_ground_truth.head()

,task_id,model_id,f1,model_rank,normalized_f1,relevance_score
0,1001,1,0.908606,19.0,0.966030,2
1,1001,2,0.900620,20.0,0.957539,2
2,1001,3,0.919784,11.0,0.977914,2
3,1001,4,0.929790,6.0,0.988553,2
4,1001,5,0.919623,13.0,0.977743,2


## 2) Build Train/Test Task Splits

We use 20% of train tasks for testing (Since we don't have the ground truth relevance scores for test tasks)

In [5]:
from sklearn.model_selection import train_test_split

In [7]:
train_task_ids = df_tasks[df_tasks["task_type"] == "train"]["task_id"].tolist()
train_task_ids, test_task_ids = train_test_split(train_task_ids, test_size=0.2, random_state=42)

In [8]:
df_ground_truth_train_only = df_ground_truth[df_ground_truth["task_id"].isin(train_task_ids)]
df_ground_truth_test_only = df_ground_truth[df_ground_truth["task_id"].isin(test_task_ids)]

## 3) Train the Baseline Ranker

For each model, compute the average rank observed on training tasks.
Lower mean rank is better

In [9]:
df_model_avg_ranking = (
    df_ground_truth_train_only.groupby("model_id")["model_rank"]
    .mean()
    .reset_index()
    .sort_values("model_rank", ascending=True)
)
df_model_avg_ranking = df_model_avg_ranking.merge(df_models[["model_id", "model_name"]], on="model_id", how="left")
df_model_avg_ranking

,model_id,model_rank,model_name
0,9,6.9625,deepset/roberta-base-squad2
1,8,7.1375,cardiffnlp/twitter-roberta-base-sentiment-latest
2,18,7.1875,roberta-base
3,15,7.2500,j-hartmann/emotion-english-distilroberta-base
4,16,7.5375,microsoft/deberta-base
5,1,7.6250,GroNLP/hateBERT
6,19,7.8375,xlm-roberta-base
7,13,7.9000,distilroberta-base
8,10,7.9375,distilbert-base-uncased
9,5,8.4250,bert-base-uncased


## Import ndcg calculator

In [10]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

from utils import ndcg_at_k

## 4) Evaluate on Test Tasks with nDCG@k

For each test task:
1. Take the baseline model order.
2. Compute nDCG at multiple cutoff values.

In [11]:
ndcg_results = []
ranked_model_ids = df_model_avg_ranking["model_id"].tolist()

for task_id in test_task_ids:
    df_task_ground_truth = df_ground_truth_test_only[df_ground_truth_test_only["task_id"] == task_id].sort_values(
        "model_rank", ascending=True
    )

    # Align per-task relevance to the baseline ranking order.
    relevance_by_model = df_task_ground_truth.set_index("model_id")["relevance_score"].to_dict()
    ranked_model_ids_relevance_scores = [relevance_by_model.get(model_id, 0) for model_id in ranked_model_ids]

    for k in [1, 3, 5]:
        ndcg_score = ndcg_at_k(ranked_model_ids_relevance_scores, k)
        ndcg_results.append({"task_id": task_id, "k": k, "ndcg_score": ndcg_score})

df_ndcg_results = pd.DataFrame(ndcg_results)
df_ndcg_results = df_ndcg_results.merge(df_tasks[["task_id", "task_criticalness"]], on="task_id", how="left")

In [12]:
df_ndcg_results

,task_id,k,ndcg_score,task_criticalness
0,1084,1,0.428571,Medium Criticalness
1,1084,3,0.615118,Medium Criticalness
2,1084,5,0.598256,Medium Criticalness
3,1054,1,0.142857,Medium Criticalness
4,1054,3,0.301260,Medium Criticalness
5,1054,5,0.374439,Medium Criticalness
6,1071,1,0.428571,Low Criticalness
7,1071,3,0.731841,Low Criticalness
8,1071,5,0.647752,Low Criticalness
9,1046,1,0.428571,High Criticalness


## Per Criticalness nDCGs

In [13]:
pd.pivot_table(
    df_ndcg_results,
    index=["task_criticalness", "task_id"],
    columns="k",
    values="ndcg_score",
)

k                                   1         3         5
task_criticalness   task_id                              
High Criticalness   1011     0.142857  0.326456  0.589168
                    1032     0.428571  0.493701  0.432966
                    1034     0.000000  0.144331  0.375366
                    1046     0.428571  0.397480  0.403002
                    1074     0.142857  0.422677  0.606785
                    1081     0.428571  0.397480  0.512520
Low Criticalness    1001     0.428571  0.428571  0.509260
                    1005     0.428571  0.690319  0.739737
                    1019     1.000000  1.000000  0.916532
                    1071     0.428571  0.731841  0.647752
                    1078     1.000000  1.000000  0.916532
Medium Criticalness 1013     0.428571  0.494932  0.575530
                    1023     0.428571  0.361532  0.407141
                    1031     0.428571  0.690319  0.840421
                    1040     0.428571  0.690319  0.739737
                    1045     1.000000  0.878583  0.774478
                    1054     0.142857  0.301260  0.374439
                    1077     1.000000  1.000000  1.000000
                    1084     0.428571  0.615118  0.598256
                    1091     0.428571  0.494932  0.676213

In [14]:
summary_table = (
    df_ndcg_results.groupby(["task_criticalness", "k"], as_index=False)["ndcg_score"]
    .mean()
    .sort_values(["task_criticalness", "k"])
)
summary_table

,task_criticalness,k,ndcg_score
0,High Criticalness,1,0.261905
1,High Criticalness,3,0.363688
2,High Criticalness,5,0.486634
3,Low Criticalness,1,0.657143
4,Low Criticalness,3,0.770146
5,Low Criticalness,5,0.745963
6,Medium Criticalness,1,0.523810
7,Medium Criticalness,3,0.614110
8,Medium Criticalness,5,0.665135
